<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week4_neural_networks/Week4_Lesson7_Lecture.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 IB CS — Week 4 · Lesson 7 (Lecture)
## Neural Networks — ANN, CNN, Backpropagation
### A4.3.8 (ANN) + A4.3.9 (CNN) + training: backpropagation and gradient descent

> ⏱ Duration: ~90 minutes (a double lesson).
> 📘 Sources: Baumgarten (Hodder IBDP) + MacKenty & Stephenson (Oxford 2025).

---

### 🎯 What we cover today (syllabus):

| Statement | Topic | Command term |
|---|---|---|
| **A4.3.8** | Structure & function of ANNs | *Outline* |
| **A4.3.9** | CNNs for spatial hierarchies in images | *Describe* |
| **+** | Backpropagation, gradient descent (training process) | part of A4.3.8 |
| **+** | Activation functions: ReLU, Sigmoid, Softmax, tanh | |

---

### ⚠️ Before we start

> This is the **hardest** topic in the IB ML section. Do **not** try to tackle backpropagation as advanced mathematics. For IB, the expected skill set is:
>
> 1. **Draw** the structure of an ANN
> 2. **Describe in words** what each layer does
> 3. **Explain** the general idea of how the network learns
>
> Baumgarten says directly: *"The training process is more complex and is far beyond the scope of your course."*
>
> 💎 **Secret #1:** IB does **not** require chain-rule or derivative formulas. It requires the **concept**.


## 🧠 Part 1 · A4.3.8 ANN Structure

> **Definition (learn this):** *An artificial neural network (ANN) is a computational model designed to mimic the human brain's interconnected neuron structure to process information. It consists of layers of nodes (called **perceptrons**) connected by pathways that transmit signals.*

### 🧩 Three types of layers

| Layer | What it does | How many neurons? |
|---|---|---|
| **Input layer** | receives features, one neuron per feature | number of features |
| **Hidden layer(s)** | extracts patterns; there may be several | arbitrary hyperparameter |
| **Output layer** | gives the final prediction | number of classes, or 1 for regression |

> 🔑 **Fully connected network** = each neuron in one layer is connected to every neuron in the next layer. This is the standard dense ANN structure.


In [ ]:
# Visualisation of ANN structure: 3 inputs → 4 hidden → 2 outputs
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 7))

# Neuron coordinates
layer_sizes = [3, 4, 2]
layer_x = [0.1, 0.5, 0.9]
colors = ['#FFD93D', '#4ECDC4', '#FF6B6B']
labels = ['Input layer\n(I1, I2, I3)', 'Hidden layer\n(H1, H2, H3, H4)', 'Output layer\n(O1, O2)']

positions = {}
for i, (size, x, color) in enumerate(zip(layer_sizes, layer_x, colors)):
    y_positions = np.linspace(0.2, 0.8, size)
    for j, y in enumerate(y_positions):
        circle = patches.Circle((x, y), 0.04, color=color, ec='black', lw=2, zorder=3)
        ax.add_patch(circle)
        positions[(i, j)] = (x, y)
        # Neuron labels
        if i == 0:
            ax.text(x - 0.07, y, f'I{j+1}', fontsize=10, ha='right', va='center')
        elif i == 1:
            ax.text(x, y + 0.06, f'H{j+1}', fontsize=9, ha='center')
        else:
            ax.text(x + 0.07, y, f'O{j+1}', fontsize=10, ha='left', va='center')

# Connections between layers
for i in range(len(layer_sizes) - 1):
    for j in range(layer_sizes[i]):
        for k in range(layer_sizes[i+1]):
            x1, y1 = positions[(i, j)]
            x2, y2 = positions[(i+1, k)]
            ax.plot([x1, x2], [y1, y2], color='gray', alpha=0.4, lw=0.7, zorder=1)

# Layer labels
for x, label in zip(layer_x, labels):
    ax.text(x, 0.05, label, ha='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Fully connected ANN: 3 → 4 → 2',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

print("📊 Parameter count:")
print("  Weights between Input(3) and Hidden(4): 3 × 4 = 12")
print("  Weights between Hidden(4) and Output(2): 4 × 2 = 8")
print("  Biases in Hidden: 4")
print("  Biases in Output: 2")
print("  TOTAL parameters: 12 + 8 + 4 + 2 = 26")


### 🔬 One perceptron under the microscope

> **Perceptron** (Baumgarten): *the data structure at the heart of an artificial neural network; it represents a single artificial neuron that takes in multiple inputs and weights, and generates an output value.*

**What happens inside one neuron (5 steps):**

| Step | Operation |
|---|---|
| 1. **Inputs** | receives values from the previous layer |
| 2. **Weights** | each input is multiplied by its weight, meaning importance |
| 3. **Summation** | all products are added together |
| 4. **Bias** | a bias term is added |
| 5. **Activation** | the result passes through an activation function |

**Formula for one neuron:**

$$y = \text{ReLU}(x_1 w_1 + x_2 w_2 + x_3 w_3 + b)$$

where: $x_i$ = inputs, $w_i$ = weights, $b$ = bias, ReLU = activation function.


In [ ]:
# Worked example from Baumgarten (p. 47): calculating one perceptron
# Inputs: 1.3, 4.2, 0.0, 2.7
# Weights: -3.1, 1.6, 2.9, 2.7
# Bias: -5.2
# Activation: ReLU

inputs = np.array([1.3, 4.2, 0.0, 2.7])
weights = np.array([-3.1, 1.6, 2.9, 2.7])
bias = -5.2

# Steps 1-3: multiplication and summation
summation = np.dot(inputs, weights)
print("Steps 1-3 — Summation:")
print("  (1.3 × -3.1) + (4.2 × 1.6) + (0.0 × 2.9) + (2.7 × 2.7)")
print(f"  = {1.3*-3.1:.2f} + {4.2*1.6:.2f} + {0.0*2.9:.2f} + {2.7*2.7:.2f}")
print(f"  = {summation:.2f}")

# Step 4: add bias
z = summation + bias
print(f"\nStep 4 — Add bias: {summation:.2f} + ({bias}) = {z:.2f}")

# Step 5: ReLU activation
output = max(0, z)  # ReLU: max(0, x)
print(f"\nStep 5 — ReLU({z:.2f}) = {output:.2f}")
print(f"\n💎 This neuron's OUTPUT = {output:.2f}")


> 💎 **Secret #2 (common in IB):** *"Describe one step of forward propagation"* **[3]** — model answer for 3/3:
> 1. *"Each neuron in the layer receives inputs from all neurons in the previous layer."*
> 2. *"Inputs are multiplied by their respective weights, summed together, and a bias is added."*
> 3. *"The result is passed through an activation function (such as ReLU) to produce the neuron's output, which is then sent to the next layer."*


## ⚡ Part 2 · Activation Functions

> **Definition:** *A mathematical function applied to the output of a neuron to determine whether it should be "activated" (output non-zero) and introduce non-linearity.*

### 🎯 Why do we need activation functions?

> Without activation, the whole network **collapses into one linear function**. Any number of linear layers is still only one linear layer.
> Activation functions add **non-linearity**, allowing the network to model **complex patterns**.

### 4 main activation functions (IB)

| Function | Formula | Output range | Where used |
|---|---|---|---|
| **ReLU** | $\max(0, x)$ | $[0, +\infty)$ | **default** for hidden layers |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | $(0, 1)$ | binary classification output |
| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $(-1, 1)$ | hidden layers, centred at 0 |
| **Softmax** | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | $(0, 1)$, sum=1 | **multi-class** output probabilities |


In [ ]:
# Visualisation of 4 activation functions
x = np.linspace(-5, 5, 200)
relu = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(x, relu, color='#FF6B6B', linewidth=2.5)
axes[0].set_title('ReLU — default for hidden layers', fontweight='bold')
axes[0].axhline(0, color='gray', linewidth=0.5); axes[0].axvline(0, color='gray', linewidth=0.5)
axes[0].grid(alpha=0.3)

axes[1].plot(x, sigmoid, color='#4ECDC4', linewidth=2.5)
axes[1].set_title('Sigmoid — binary output', fontweight='bold')
axes[1].axhline(0, color='gray', linewidth=0.5); axes[1].axhline(1, color='gray', linewidth=0.5, ls='--')
axes[1].axvline(0, color='gray', linewidth=0.5)
axes[1].grid(alpha=0.3)

axes[2].plot(x, tanh, color='#FFD93D', linewidth=2.5)
axes[2].set_title('Tanh — hidden, centred at 0', fontweight='bold')
axes[2].axhline(0, color='gray', linewidth=0.5); axes[2].axvline(0, color='gray', linewidth=0.5)
axes[2].axhline(1, color='gray', linewidth=0.5, ls='--')
axes[2].axhline(-1, color='gray', linewidth=0.5, ls='--')
axes[2].grid(alpha=0.3)

# Softmax for 3 classes
logits = np.array([[xi-1, xi, xi+1] for xi in x])
softmax_vals = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
axes[3].plot(x, softmax_vals[:, 0], label='Class 1', linewidth=2)
axes[3].plot(x, softmax_vals[:, 1], label='Class 2', linewidth=2)
axes[3].plot(x, softmax_vals[:, 2], label='Class 3', linewidth=2)
axes[3].set_title('Softmax — multi-class output', fontweight='bold')
axes[3].legend(fontsize=8); axes[3].grid(alpha=0.3)
axes[3].set_ylabel('Probability')

plt.suptitle('4 main activation functions in the IB syllabus',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()


### 🆚 ReLU vs Sigmoid — why did ReLU become standard?

| Criterion | ReLU | Sigmoid |
|---|---|---|
| Computation speed | **very fast**: just max | slower: uses exp |
| Vanishing gradient | **rare** | **common problem** |
| Output range | [0, +∞) | (0, 1) |
| Where used | hidden layers | output for binary tasks |

> 💎 **Secret #3:** *"Why is ReLU preferred over Sigmoid in deep networks?"* **[3]** — answer for 3/3:
> 1. **Vanishing gradient problem**: sigmoid has very small gradients for large |x|, so backpropagation fades in deep networks
> 2. **Computational efficiency**: ReLU is simply `max(0, x)`, while sigmoid needs `exp()`
> 3. **Biological plausibility**: ReLU imitates the all-or-nothing behaviour of biological neurons


## 🔄 Part 3 · Forward & Backpropagation (training)

### Neural-network training cycle — 4 steps (MacKenty/Stephenson)

| Step | What happens |
|---|---|
| 1. **Forward propagation** | data flows from input to output and a prediction is calculated |
| 2. **Calculate loss** | compare prediction with the true value using a loss function |
| 3. **Backpropagation** | error is propagated backward and gradients are calculated for each weight |
| 4. **Update weights** | weights are updated with gradient descent and learning rate |

Repeat for **N epochs**. One epoch means one full pass through the training set.


In [ ]:
# Visualisation of the training cycle
fig, ax = plt.subplots(figsize=(13, 6))

# 4 stages as a cycle
import matplotlib.patches as mpatches
stages = [
    ('1. Forward\nPropagation', 0.2, 0.7, '#4ECDC4',
     'Input → Hidden → Output\ny_pred = f(W·x + b)'),
    ('2. Calculate\nLoss', 0.55, 0.85, '#FFD93D',
     'Loss = MSE / Cross-entropy\nL = (y_pred - y_true)²'),
    ('3. Back-\npropagation', 0.85, 0.7, '#FF6B6B',
     'Compute ∂L/∂w for each weight\n(chain rule)'),
    ('4. Update\nWeights', 0.55, 0.25, '#A78BFA',
     'w_new = w_old - lr · ∂L/∂w\n(gradient descent)'),
]

for label, x, y, color, formula in stages:
    box = mpatches.FancyBboxPatch((x-0.08, y-0.06), 0.16, 0.12,
                                    boxstyle="round,pad=0.02",
                                    facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(x, y - 0.13, formula, ha='center', va='center', fontsize=8, style='italic')

# Arrows between stages
arrow_props = dict(arrowstyle='->', lw=2, color='black')
ax.annotate('', xy=(0.47, 0.82), xytext=(0.30, 0.74), arrowprops=arrow_props)
ax.annotate('', xy=(0.77, 0.74), xytext=(0.63, 0.82), arrowprops=arrow_props)
ax.annotate('', xy=(0.63, 0.30), xytext=(0.79, 0.62), arrowprops=arrow_props)
ax.annotate('', xy=(0.28, 0.62), xytext=(0.47, 0.30), arrowprops=arrow_props)

ax.text(0.5, 0.5, 'EPOCH\n(full cycle)',
        ha='center', va='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle="round", facecolor='lightgray', edgecolor='black'))

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Neural-network training cycle (1 epoch)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


### 🎯 Backpropagation — the key idea in one paragraph

> **Definition (Baumgarten):** *Backpropagation is the most commonly used technique for training ANNs. The gradient of the loss function is calculated and used to update parameters such as weights, in the opposite direction of the gradient to reduce the overall error.*

**In simple terms:**
1. The network makes a prediction (forward pass).
2. We calculate how far the prediction is from the correct answer (**loss**).
3. The error is **propagated backward** through the layers: which weights caused the error most?
4. We adjust weights in the **opposite direction** of the gradient using gradient descent.

> 💎 **Secret #4 (common in IB):** *"Define backpropagation and outline its role in learning"* **[3]** — Baumgarten Review #18:
> 1. **Definition** (1 mark): backpropagation sends prediction error backward from output to input layers.
> 2. **Calculates gradients** (1 mark): it calculates how much each weight contributed to the error using the gradient of the loss function.
> 3. **Updates weights** (1 mark): gradients update weights through gradient descent, reducing loss over multiple epochs.

---

### 📉 Gradient Descent + Learning Rate

> **Gradient descent** is an optimisation algorithm that moves downhill on the loss surface.
>
> **Learning rate (η)** controls the step size.

**Learning-rate trade-off:**
- **Too large** (η = 1.0) → jumps over the minimum; loss may bounce or diverge
- **Too small** (η = 0.0001) → training is **very slow**
- **Good range** (η ≈ 0.001-0.01) → stable and fast learning


In [ ]:
# Visualisation: gradient descent and how learning rate affects convergence
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Parabolic loss function
def loss(w):
    return (w - 3)**2 + 1

def gradient(w):
    return 2*(w - 3)

w_range = np.linspace(-2, 8, 100)
losses = [loss(w) for w in w_range]

learning_rates = [0.05, 0.5, 1.1]
titles = ['Small LR (η=0.05)\nslow but stable',
          'Optimal LR (η=0.5)\nfast and stable',
          'Large LR (η=1.1)\ndiverges']
colors = ['steelblue', 'green', 'red']

for ax, lr, title, color in zip(axes, learning_rates, titles, colors):
    ax.plot(w_range, losses, 'k-', linewidth=2, alpha=0.5)

    # Simulate gradient descent
    w = -1.5  # starting point
    path_w = [w]
    path_l = [loss(w)]
    for _ in range(15):
        w = w - lr * gradient(w)
        path_w.append(w)
        path_l.append(loss(w))
        if abs(w) > 10:  # diverging case
            break

    ax.plot(path_w, path_l, 'o-', color=color, markersize=8, linewidth=1.5,
            label=f'{len(path_w)-1} steps')
    ax.scatter([path_w[0]], [path_l[0]], s=200, color=color, marker='*',
               edgecolor='black', linewidth=2, zorder=5, label='start')
    ax.scatter([3], [1], s=200, color='gold', marker='X',
               edgecolor='black', linewidth=2, zorder=5, label='minimum')

    ax.set_xlabel('weight w'); ax.set_ylabel('Loss')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.set_ylim(-2, 50)

plt.tight_layout(); plt.show()


### ⚠️ Overfitting in neural networks

> **Common mistake (Baumgarten):** *"Overcomplicating the model architecture can lead to overfitting and high computational costs. Start with a simple architecture and gradually increase complexity, if necessary."*

**Signs of overfitting:**
- Training accuracy >> test accuracy, meaning a large gap
- Training loss falls, but validation loss **rises**
- The network has **many parameters** but only **limited data**

**Methods to reduce it:**
1. **Dropout** — randomly switch off neurons during training
2. **L2 regularisation** — penalise large weights
3. **Early stopping** — stop when validation loss stops decreasing
4. **Data augmentation** — artificially increase the dataset with rotations, noise, and similar changes
5. **Smaller architecture** — fewer layers or neurons


## 🖼️ Part 4 · A4.3.9 Convolutional Neural Networks (CNN)

> **Definition (Baumgarten):** *A CNN extends ANN architecture by using additional layers of calculations prior to processing data through a fully connected ANN. It applies **convolution** — a mathematical operation that uses filtering functions to compute distinctive features from images.*

### 🎯 Why use CNN if ANN exists?

> A 100×100 pixel RGB image has **30,000 values**.
> Passing all of them into a plain ANN would require **millions of parameters**, causing overfitting and slow training.
>
> A **CNN** first compresses the image through convolution and pooling, **extracting important patterns**.

### 🏗️ CNN architecture (5 layer types)

| Layer | What it does | Why it matters |
|---|---|---|
| **Input** | receives raw pixels | height × width × channels |
| **Convolutional** | applies filters, or kernels, to create feature maps | detects edges, textures, patterns |
| **Activation** (ReLU) | adds non-linearity | important after each convolution |
| **Pooling** | reduces feature-map size | reduces computation and overfitting |
| **Fully connected** | classic ANN at the end | final classification |
| **Output** (softmax) | class probabilities | probabilities sum to 1 |


In [ ]:
# Visualisation of CNN architecture
fig, ax = plt.subplots(figsize=(15, 5))

stages = [
    ('Input\n28×28×1', 0.05, '#FFD93D'),
    ('Conv\n3×3 kernels', 0.20, '#4ECDC4'),
    ('ReLU', 0.32, '#FF6B6B'),
    ('Max Pool\n2×2', 0.42, '#A78BFA'),
    ('Conv\n3×3 kernels', 0.55, '#4ECDC4'),
    ('ReLU', 0.65, '#FF6B6B'),
    ('Max Pool', 0.74, '#A78BFA'),
    ('Flatten', 0.83, '#FFD93D'),
    ('Fully\nConnected', 0.91, '#FFA07A'),
    ('Softmax\n→ 10 classes', 0.99, '#90EE90'),
]

for label, x, color in stages:
    box = mpatches.FancyBboxPatch((x-0.04, 0.4), 0.07, 0.25,
                                    boxstyle="round,pad=0.01",
                                    facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, 0.525, label, ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows
for i in range(len(stages) - 1):
    x1 = stages[i][1] + 0.025
    x2 = stages[i+1][1] - 0.045
    ax.annotate('', xy=(x2, 0.525), xytext=(x1, 0.525),
                arrowprops=dict(arrowstyle='->', lw=1.5))

ax.text(0.05, 0.85, '📷 Image', ha='center', fontsize=10, style='italic')
ax.text(0.30, 0.85, '🔍 Feature extraction', ha='center', fontsize=10, style='italic')
ax.text(0.65, 0.85, '🔍 Higher features', ha='center', fontsize=10, style='italic')
ax.text(0.91, 0.85, '🎯 Classification', ha='center', fontsize=10, style='italic')

ax.text(0.30, 0.20, 'Layer 1: Edges, simple shapes',
        ha='center', fontsize=9, color='gray')
ax.text(0.65, 0.20, 'Layer 2: Complex patterns (eyes, fur)',
        ha='center', fontsize=9, color='gray')

ax.set_xlim(0, 1.05); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Typical CNN architecture for image classification',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()


### 🔍 Convolutional Layer — the most interesting part

> **Kernel (filter)** — a small matrix, often 3×3 or 5×5, that slides over an image and **extracts patterns**.

**Examples of known filters (Baumgarten, p. 58):**

| Filter | Purpose |
|---|---|
| Vertical edge detector | finds vertical edges |
| Horizontal edge detector | finds horizontal edges |
| Sharpening | increases contrast |
| Gaussian blur | blurs and reduces noise |

> 💎 **Secret #5:** *"Strength of CNNs lies in their ability to LEARN the most appropriate filters for a given task through backpropagation"* (Baumgarten).
> CNNs do **not** simply use ready-made filters. They **learn** filters during training.

**How convolution works:**
- A 3×3 filter slides across the image
- At each position, it calculates a **dot product**
- The result becomes one pixel in a **feature map**


In [ ]:
# Demo: apply a vertical edge detector to a simple image
from scipy.signal import convolve2d

# Simple image: left half bright, right half dark
image = np.zeros((20, 20))
image[:, :10] = 1.0  # bright half
# Add noise
image += np.random.normal(0, 0.05, image.shape)

# Vertical edge detection kernel (Sobel)
kernel = np.array([
    [1, 0, -1],
    [2, 0, -2],
    [1, 0, -1]
])

feature_map = convolve2d(image, kernel, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image\n(20×20)', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(kernel, cmap='RdBu_r', vmin=-2, vmax=2)
axes[1].set_title('Vertical Edge Kernel\n(3×3)', fontweight='bold')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{kernel[i, j]:+d}',
                    ha='center', va='center', fontsize=14, fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])

axes[2].imshow(np.abs(feature_map), cmap='hot')
axes[2].set_title('Feature Map after Convolution\n(vertical edge is visible)',
                  fontweight='bold')
axes[2].axis('off')

plt.tight_layout(); plt.show()
print("💎 The kernel highlighted the vertical boundary between the bright and dark regions.")


### 🌊 Pooling Layer

> **Why use it?** (3 reasons from Baumgarten):
> 1. **Reduce dimensions** → fewer parameters → faster training
> 2. **Reduce overfitting** → less focus on tiny details
> 3. **Translation invariance** → object can be recognised in different positions

**Two types of pooling:**
- **Max pooling** — takes the **maximum** from a window, such as 2×2, keeping the strongest features
- **Average pooling** — takes the **mean**, smoother and discards less information

> 💎 **Max pooling** is the most common. For IB *"Outline ONE method of pooling"*, max pooling is usually the clearest answer.


In [ ]:
# Demo: max pooling
demo_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [3, 1, 7, 8],
    [2, 4, 5, 6],
])

# Max pooling 2×2
pooled = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        pooled[i, j] = demo_map[2*i:2*i+2, 2*j:2*j+2].max()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(demo_map, cmap='Blues')
axes[0].set_title('Feature Map (4×4)', fontweight='bold')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{demo_map[i,j]}', ha='center', va='center',
                     fontsize=16, fontweight='bold')
# Draw boundaries for 2×2 regions
for i in [0, 2]:
    for j in [0, 2]:
        rect = mpatches.Rectangle((j-0.5, i-0.5), 2, 2, linewidth=3,
                                    edgecolor='red', facecolor='none')
        axes[0].add_patch(rect)
axes[0].set_xticks([]); axes[0].set_yticks([])

axes[1].imshow(pooled, cmap='Blues')
axes[1].set_title('After Max Pooling 2×2 (→ 2×2)', fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{int(pooled[i,j])}', ha='center', va='center',
                     fontsize=20, fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout(); plt.show()
print("💎 Max pooling takes the maximum from each 2×2 window. 4×4 → 2×2.")


### 🎯 CNN vs ANN — comparison table for IB

| Criterion | ANN | CNN |
|---|---|---|
| Input data | flat feature vector | **image** matrix |
| Uses convolution? | no | **yes** |
| Uses pooling? | no | **yes** |
| Preserves spatial structure? | **no**, it is lost when flattened | **yes** |
| Main application | tabular data, regression | **images**, video, audio |
| Number of parameters | very high | **lower**, due to shared weights |

> 💎 **Secret #6:** *"Describe how CNNs adaptively learn spatial hierarchies of features in images"* is the exact A4.3.9 syllabus wording. Model full-mark answer:
>
> 1. *"Early layers detect simple features like edges and textures."*
> 2. *"Middle layers combine these into more complex shapes (eyes, ears)."*
> 3. *"Deeper layers recognize entire objects (faces of cats vs dogs)."*
> 4. *"This hierarchical learning happens automatically through backpropagation, without manual feature engineering."*


## 📝 Part 5 · Mini Exam Practice (in class)

### Question 1 (Baumgarten Review #17, p. 39)
> *A financial institution employs an artificial neural network to predict loan default risk based on customer profiles.*
>
> **a)** *Identify* one type of layer often used in neural networks and its purpose. **[2]**
> **b)** *Describe* why overfitting might be a concern in neural networks. **[3]**
> **c)** *Describe* how a neural network can be trained to minimize prediction error in this financial context. **[3]**

> 💎 **Breakdown for (c) [3]:**
> 1. **Forward propagation** produces an initial prediction (1 mark)
> 2. **Loss calculation**, such as binary cross-entropy for default/no-default (1 mark)
> 3. **Backpropagation** + gradient descent update weights to minimise loss over epochs (1 mark)

---

### Question 2 (Baumgarten Review #18, p. 39)
> *An energy company uses a neural network to forecast electricity demand based on weather conditions and historical usage.*
>
> **a) i)** *Identify* whether this would be regression or classification based. **[1]**
> **a) ii)** *Outline* the significance of that on its design. **[2]**
> **b)** *Outline* one type of activation function used in neural networks and its purpose. **[2]**
> **c)** *Outline* which activation function would most likely be suitable for the **output layer** in this scenario, and why. **[2]**
> **d)** *Describe* why deep neural networks might be more effective than shallow networks for this task. **[3]**
> **e)** *Define* "backpropagation" and *outline* its role in learning. **[3]**

> 💎 **Breakdown for (a):** Regression, because electricity demand is a continuous value.
> 💎 **Breakdown for (c):** **Linear** or possibly **ReLU** output; regression needs a continuous output. **Not** sigmoid/softmax, because they restrict the range.
> 💎 **Breakdown for (d):** Deep networks can extract patterns hierarchically: early layers learn simple relations, deeper layers combine weekday + season + temperature for a more accurate forecast.


## ✅ Checklist after Lesson 7

By the next lesson, the MNIST workshop, you should be able to:

### A4.3.8 ANN
- [ ] Know **3 layer types**: input, hidden, output, and their roles
- [ ] Describe the **5 steps** inside one perceptron
- [ ] Know **4 activation functions** and when to use each
- [ ] **Calculate** a forward pass for one neuron from a worked example
- [ ] Explain **backpropagation** in 3 ideas
- [ ] Understand the role of **learning rate** and its trade-off

### A4.3.9 CNN
- [ ] Know **5 CNN layer types** and their roles
- [ ] Explain what a **kernel/filter** is
- [ ] Know what a **feature map** is
- [ ] Distinguish **max pooling** and **average pooling**
- [ ] Explain **spatial hierarchies**, the exact A4.3.9 syllabus wording

### General
- [ ] Understand **CNN vs ANN** and when to use each
- [ ] Know **overfitting** in neural networks and 2-3 ways to reduce it

---

### 📚 Homework (see `Week4_HW1_Theory.ipynb`)

1. Draw an ANN diagram for a task with 3 inputs → 2 outputs
2. Calculate a forward pass **by hand** for one neuron
3. Compare ReLU vs Sigmoid
4. IB-style questions on A4.3.8 and A4.3.9

---

> 🎓 **Final secret of Lesson 7:**
> Neural networks scare students because of backpropagation mathematics.
> But **IB does NOT ask for that mathematics**, as Baumgarten states: *"beyond the scope of your course"*.
> IB asks for **structure**, **layer roles**, **why activation functions matter**, and **the general idea of forward/back propagation**.
>
> 💎 Learn the terms and the marks become much more predictable.
